# 02 - Data Preprocessing

- Run reusable preprocessing from `src/`.
- Save cleaned product and comment parquet files.

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.rag.preprocessing.datasets import preprocess_datasets

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

## Run preprocessing

- Build processed datasets.

In [2]:
result = preprocess_datasets(
    products_path=RAW_DIR / "digikala-products.csv",
    comments_path=RAW_DIR / "digikala-comments.csv",
    output_dir=PROCESSED_DIR,
)

products = result["products"]
comments = result["comments"]

print("Products:", products.shape)
print("Comments:", comments.shape)
print("Saved:", result["products_path"])
print("Saved:", result["comments_path"])

Products: (960367, 12)
Comments: (6153060, 16)
Saved: /home/ali/Desktop/projects/digikala-ai-assistant/data/processed/products_clean.parquet
Saved: /home/ali/Desktop/projects/digikala-ai-assistant/data/processed/comments_clean.parquet


## Sanity checks

- Validate duplicates, rating ranges, product links, and output samples.

In [3]:
checks = {
    "product_exact_duplicates": int(products.duplicated().sum()),
    "comment_exact_duplicates": int(comments.duplicated().sum()),
    "comment_duplicate_ids": int(comments["id"].duplicated().sum()),
    "comment_rate_gt_5": int((comments["rate"] > 5).sum()),
    "unmatched_comment_products": int(
        (~comments["product_id"].isin(products["id"])).sum()
    ),
}

pd.Series(checks)

product_exact_duplicates      0
comment_exact_duplicates      0
comment_duplicate_ids         0
comment_rate_gt_5             0
unmatched_comment_products    0
dtype: int64

In [4]:
display(
    products[
        ["id", "title_fa", "Brand", "Price"]
    ].head()
)

display(
    comments[
        [
            "id",
            "product_id",
            "rate",
            "body",
            "created_at_gregorian",
        ]
    ].head()
)

print(
    "Comment date range:",
    comments["created_at_gregorian"].min(),
    "→",
    comments["created_at_gregorian"].max(),
)

,id,title_fa,Brand,Price
0,7096438,آبسلانگ مدل s5 بسته 250 عددی,متفرقه,634800.0
1,2845119,آبسلانگ مدل M-1 بسته 400 عددی,متفرقه,818800.0
2,6117745,آبسلانگ مدل m50 مجموعه 500 عددی,متفرقه,920000.0
3,1912926,استند ابسلانگ مدل S01,متفرقه,1100000.0
4,6335462,آبسلانگ نوری تسلامد مدل All-in-One,تسلا مد,1530000.0


,id,product_id,rate,body,created_at_gregorian
0,14144758,1075274,5.0,عالیه مخصوصا طرحش عکس مجموعه هری پاترم رو میزا...,2020-11-30
1,41782279,6081008,3.0,توجه فرمایید که عکس اول واسه ۲ هفته پیشه و عکس...,2022-11-12
2,49569443,10545754,1.0,متاسفانه از سال ۹۶ مشتری دیجیکالا هستم.بالای ۴...,2023-05-24
3,43932524,4153832,5.0,تصمیم خرید کنسول برای منِ 32 ساله با هزینه شخص...,2023-01-05
4,21396693,2185657,3.0,اول لاک معمولی میزدم و بعد از خشک شدن کامل این...,2021-05-30


Comment date range: 2016-07-13 00:00:00 → 2023-10-18 00:00:00
